# Transient model: observation preparation

**Alwar groundwater study · 14-block domain (LGD 574–587)**

Restores the MODFLOW 6 + FloPy environment and prepares the groundwater observations for the transient model: QC-passed observation table (Step 1) and the well spatial database (Step 2A).

| | |
|---|---|
| **Reads** | `Rajasthan_gw_data/…` cleaned groundwater tables |
| **Writes** | `gw_framework/MODFLOW_Alwar/` (Step 1 and Step 2 CSVs) |
| **Cells** | 4 code cells, copied unchanged from `alwargwd_data_analysis_2_2.ipynb` (outputs cleared) |

**How to run:** run the set-up cell, then each section from top to bottom. Every cell reads its inputs from Drive, except where a label says it needs the cell above.

Cell labels: `[orig #N]` = position in the original notebook · ✅ ran OK · ⚠ last run ended with an error · ⏸ interrupted · • output not saved.

**Contents**
- 1. Environment
- 2. Observation preparation

In [ ]:
# ── SETUP — run this cell first ─────────────────────────────────────────────
import os, sys, json, subprocess
from google.colab import drive
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

CFG = "/content/drive/MyDrive/gw_framework/repo_config.json"   # remembers your repository
if os.path.isfile(CFG):
    REPO = json.load(open(CFG))["repo"]
else:
    REPO = input("GitHub repository (username/repo-name): ").strip().strip("/")
    json.dump({"repo": REPO}, open(CFG, "w"))

REPO_DIR = "/content/" + REPO.split("/")[-1]
try:                                   # private repository: add a Colab secret named GH_TOKEN
    from google.colab import userdata
    TOKEN = userdata.get("GH_TOKEN")
except Exception:
    TOKEN = None
URL = f"https://{TOKEN + '@' if TOKEN else ''}github.com/{REPO}.git"
if os.path.isdir(REPO_DIR + "/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=False)
elif subprocess.run(["git", "clone", "-q", URL, REPO_DIR]).returncode != 0:
    os.remove(CFG)
    raise RuntimeError(f"Could not clone '{REPO}'. Check the name (and GH_TOKEN if private), then run this cell again.")
sys.path.insert(0, REPO_DIR + "/src")

from alwar_gw import colab
PATHS = colab.setup(packages=["flopy", "geopandas", "rasterio", "statsmodels", "xarray", "netCDF4"], modflow=False)
del TOKEN, URL


## 1. Environment

**[orig #250]** Install FloPy and GIS packages; print versions  
<sub>• output not saved in the source notebook</sub>

In [ ]:

!pip -q install flopy geopandas rasterio pyproj shapely scipy statsmodels xarray netCDF4

import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import flopy
import geopandas as gpd

print("Python:", sys.version)
print("FloPy version:", flopy.__version__)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Environment restored.")


**[orig #252]** Step 1 — restore the MODFLOW 6 + FloPy environment  
<sub>• output not saved in the source notebook</sub>

In [ ]:

# ============================================================
# STEP 1: RESTORE MODFLOW 6 + FLOPY ENVIRONMENT
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

!pip -q install flopy geopandas rasterio xarray netCDF4 \
    scipy pandas numpy matplotlib pyproj shapely

import os
import sys
import flopy
import numpy as np
import pandas as pd

# Research directory
PROJECT = "/content/drive/MyDrive/gw_framework"

# MODFLOW workspace
MODEL_DIR = os.path.join(
    PROJECT, "MODFLOW_Alwar"
)

os.makedirs(MODEL_DIR, exist_ok=True)

# Check MODFLOW executable
mf6_exe = os.path.join(MODEL_DIR, "mf6")

if not os.path.exists(mf6_exe):
    print("Installing MODFLOW 6...")
    flopy.utils.get_modflow(
        MODEL_DIR,
        repo="modflow6",
        release_id="latest"
    )
else:
    print("MODFLOW executable already exists.")

print("\n========== ENVIRONMENT STATUS ==========")
print("Python:", sys.version)
print("FloPy:", flopy.__version__)
print("Project:", PROJECT)
print("Model directory:", MODEL_DIR)
print("MODFLOW executable:", os.path.exists(mf6_exe))
print("========================================")


## 2. Observation preparation

**[orig #253]** **MAIN** Step 1 — groundwater observation preparation for the transient model → `MODFLOW_Alwar/`  
<sub>• output not saved in the source notebook</sub>

In [ ]:

# ============================================================
# ALWAR TRANSIENT GROUNDWATER MODEL
# STEP 1: GROUNDWATER OBSERVATION PREPARATION
# MODFLOW 6 + FLOPY
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import re
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. PROJECT PATHS
# ------------------------------------------------------------

ROOT = "/content/drive/MyDrive/gw_framework"

INPUT_FILE = os.path.join(
    ROOT, "data/Rajasthan_gw_data/Alwar.xlsx"
)

OUT = os.path.join(
    ROOT, "MODFLOW_Alwar", "01_Groundwater_Observations"
)

os.makedirs(OUT, exist_ok=True)

assert os.path.exists(INPUT_FILE), (
    f"Alwar workbook not found: {INPUT_FILE}"
)

# ------------------------------------------------------------
# 2. READ ORIGINAL DATA
# ------------------------------------------------------------

df = pd.read_excel(INPUT_FILE)

print("Original rows:", len(df))
print("Original columns:", len(df.columns))

# Standardize column names
df.columns = [str(c).strip() for c in df.columns]

# ------------------------------------------------------------
# 3. IDENTIFY REQUIRED COLUMNS
# ------------------------------------------------------------

required = [
    "Latitude",
    "Longitude",
    "RL (m)",
    "Total_Depth_bgl"
]

for col in required:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

# Use WIMS_ID as primary identifier
if "WIMS_ID" in df.columns:
    df["Well_ID"] = df["WIMS_ID"].astype("string")
elif "ROLTA_ID/UID" in df.columns:
    df["Well_ID"] = df["ROLTA_ID/UID"].astype("string")
else:
    raise ValueError("No suitable well identifier found.")

# ------------------------------------------------------------
# 4. EXTRACT PRE- AND POST-MONSOON MEASUREMENTS
# ------------------------------------------------------------

records = []
unrecognized = []

for col in df.columns:

    match = re.fullmatch(
        r"WL[_\s]*(Pre|Post)[_\s]*(\d{2,4})",
        col,
        flags=re.IGNORECASE
    )

    if not match:
        continue

    season = match.group(1).lower()
    year_value = int(match.group(2))

    if year_value < 100:
        year = 2000 + year_value
    else:
        year = year_value

    for idx, row in df.iterrows():

        raw = row[col]

        if pd.isna(raw):
            continue

        value = pd.to_numeric(raw, errors="coerce")

        # Preserve nonnumeric observations for review
        if pd.isna(value):
            unrecognized.append({
                "source_row": idx,
                "Well_ID": row["Well_ID"],
                "column": col,
                "original_value": str(raw)
            })
            continue

        records.append({
            "Well_ID": row["Well_ID"],
            "Year": year,
            "Season": season,
            "DTW_m": float(value),
            "Latitude": row["Latitude"],
            "Longitude": row["Longitude"],
            "Ground_RL_m": row["RL (m)"],
            "Well_Depth_m": row["Total_Depth_bgl"],
            "Block": row.get("Block"),
            "Formation": row.get("Hyd_Formation"),
            "Source_Column": col,
            "Source_Row": idx
        })

obs = pd.DataFrame(records)

if obs.empty:
    raise ValueError("No numerical groundwater observations extracted.")

# ------------------------------------------------------------
# 5. NUMERIC CONVERSION
# ------------------------------------------------------------

for col in [
    "DTW_m", "Latitude", "Longitude",
    "Ground_RL_m", "Well_Depth_m"
]:
    obs[col] = pd.to_numeric(
        obs[col], errors="coerce"
    )

# ------------------------------------------------------------
# 6. CALCULATE GROUNDWATER HEAD
# ------------------------------------------------------------

obs["Groundwater_Head_m"] = (
    obs["Ground_RL_m"] - obs["DTW_m"]
)

# ------------------------------------------------------------
# 7. QUALITY CONTROL FLAGS
# ------------------------------------------------------------

obs["QC_Flag"] = "PASS"

obs.loc[
    obs["DTW_m"] < 0,
    "QC_Flag"
] = "REVIEW_NEGATIVE_DTW"

obs.loc[
    obs["DTW_m"] <= 0,
    "QC_Flag"
] = "REVIEW_NONPOSITIVE_DTW"

obs.loc[
    obs["Well_Depth_m"].notna() &
    (obs["DTW_m"] > obs["Well_Depth_m"]),
    "QC_Flag"
] = "REVIEW_BELOW_WELL_BOTTOM"

obs.loc[
    obs["Latitude"].isna() |
    obs["Longitude"].isna() |
    obs["Ground_RL_m"].isna(),
    "QC_Flag"
] = "MISSING_SPATIAL_OR_ELEVATION_DATA"

obs.loc[
    ~obs["Latitude"].between(6, 38) |
    ~obs["Longitude"].between(66, 101),
    "QC_Flag"
] = "INVALID_COORDINATES"

# No observations are automatically deleted.

# ------------------------------------------------------------
# 8. IDENTIFY DUPLICATES
# ------------------------------------------------------------

obs["Duplicate_Key"] = (
    obs["Well_ID"].astype(str) + "_" +
    obs["Year"].astype(str) + "_" +
    obs["Season"].astype(str)
)

duplicate_records = obs[
    obs.duplicated("Duplicate_Key", keep=False)
].copy()

# ------------------------------------------------------------
# 9. EXPORT OUTPUTS
# ------------------------------------------------------------

obs.to_csv(
    os.path.join(OUT, "Alwar_GW_Observations_All.csv"),
    index=False
)

obs[obs["QC_Flag"] == "PASS"].to_csv(
    os.path.join(OUT, "Alwar_GW_Observations_QC_Pass.csv"),
    index=False
)

obs[obs["QC_Flag"] != "PASS"].to_csv(
    os.path.join(OUT, "Alwar_GW_Observations_Review.csv"),
    index=False
)

duplicate_records.to_csv(
    os.path.join(OUT, "Duplicate_Observations.csv"),
    index=False
)

pd.DataFrame(unrecognized).to_csv(
    os.path.join(OUT, "NonNumeric_Observations.csv"),
    index=False
)

# ------------------------------------------------------------
# 10. SUMMARY
# ------------------------------------------------------------

print("\n==========================================")
print("STEP 1 COMPLETED")
print("==========================================")

print("Total extracted observations:", len(obs))
print("Unique wells:", obs["Well_ID"].nunique())
print("Years:", sorted(obs["Year"].unique()))
print("\nSeasonal observations:")
print(obs.groupby(["Year", "Season"]).size())

print("\nQC summary:")
print(obs["QC_Flag"].value_counts())

print("\nDuplicate records:", len(duplicate_records))
print("Nonnumeric values:", len(unrecognized))

print("\nOutput directory:")
print(OUT)

print("\nIMPORTANT:")
print("No observations were interpolated or deleted.")
print("Original Excel workbook remains unchanged.")
print("Campaign dates have not been invented.")


**[orig #254]** **MAIN** Step 2A — well spatial database from the Step 1 outputs  
<sub>• output not saved in the source notebook</sub>

In [ ]:

# ============================================================
# STEP 2A: ALWAR GROUNDWATER WELL SPATIAL DATABASE
# Uses completed Step 1 outputs; does NOT repeat extraction.
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os, glob, re, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

BASE = "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar"
IN_DIR = os.path.join(BASE, "01_Groundwater_Observations")
OUT_DIR = os.path.join(BASE, "02_Spatial_Hydrogeology")

os.makedirs(OUT_DIR, exist_ok=True)

if not os.path.isdir(IN_DIR):
    raise FileNotFoundError(f"Step 1 folder not found: {IN_DIR}")

# ------------------------------------------------------------
# 1. Find Step 1 CSV files and identify the observation table
# ------------------------------------------------------------
csv_files = glob.glob(os.path.join(IN_DIR, "*.csv"))

print("CSV files found:")
for f in csv_files:
    print(" -", os.path.basename(f))

if not csv_files:
    raise FileNotFoundError("No CSV files found in the Step 1 folder.")

def norm(c):
    return re.sub(r"[^a-z0-9]+", "", str(c).lower())

def find_col(columns, aliases):
    lookup = {norm(c): c for c in columns}
    for a in aliases:
        if norm(a) in lookup:
            return lookup[norm(a)]
    return None

candidates = []

for f in csv_files:
    try:
        temp = pd.read_csv(f, nrows=5)
        cols = temp.columns.tolist()

        lat = find_col(cols, ["Latitude", "Lat"])
        lon = find_col(cols, ["Longitude", "Lon", "Long"])
        qc = find_col(cols, ["QC_Flag", "QC Flag", "QC_Status"])

        if lat and lon:
            # Prefer the main/all-observations table with QC information.
            score = 2 if qc else 1
            size = os.path.getsize(f)
            candidates.append((score, size, f, cols))
    except Exception as e:
        print("Could not inspect:", os.path.basename(f), e)

if not candidates:
    raise ValueError(
        "Could not identify a CSV with Latitude and Longitude columns. "
        "Check the filenames and column names printed above."
    )

candidates.sort(reverse=True)
OBS_FILE = candidates[0][2]

print("\nSelected observation file:", OBS_FILE)

df = pd.read_csv(OBS_FILE, low_memory=False)
print("Observation table shape:", df.shape)
print("Columns:", df.columns.tolist())

# ------------------------------------------------------------
# 2. Identify important fields
# ------------------------------------------------------------
lat_col = find_col(df.columns, ["Latitude", "Lat"])
lon_col = find_col(df.columns, ["Longitude", "Lon", "Long"])

id_col = find_col(
    df.columns,
    ["Well_ID", "WellID", "Well_Key", "WIMS_ID",
     "ROLTA_ID/UID", "ROLTA_ID", "UID", "SSN"]
)

block_col = find_col(df.columns, ["Block", "Block_Name"])
village_col = find_col(df.columns, ["Village"])
formation_col = find_col(
    df.columns,
    ["Hyd_Formation", "Hydrogeological_Formation",
     "Aquifer", "Formation"]
)
qc_col = find_col(df.columns, ["QC_Flag", "QC Flag", "QC_Status"])

if not lat_col or not lon_col:
    raise ValueError("Latitude/Longitude columns are missing.")

if not id_col:
    raise ValueError(
        "No well identifier found. Please inspect the printed columns."
    )

print("\nIdentified fields:")
print("Well ID:", id_col)
print("Latitude:", lat_col)
print("Longitude:", lon_col)
print("Block:", block_col)
print("Village:", village_col)
print("Formation:", formation_col)
print("QC:", qc_col)

# ------------------------------------------------------------
# 3. Convert coordinates and retain all records for audit
# ------------------------------------------------------------
df[lat_col] = pd.to_numeric(df[lat_col], errors="coerce")
df[lon_col] = pd.to_numeric(df[lon_col], errors="coerce")

valid_coord = (
    df[lat_col].between(20, 35) &
    df[lon_col].between(65, 90)
)

# This is a broad India coordinate sanity check, not an
# Alwar district boundary test.
df["Coordinate_Check"] = np.where(
    valid_coord, "VALID_BROAD_RANGE", "REVIEW_COORDINATE"
)

df.to_csv(
    os.path.join(OUT_DIR, "Step2_All_Observation_Records_With_Coordinate_Check.csv"),
    index=False
)

# ------------------------------------------------------------
# 4. Create one row per well
# ------------------------------------------------------------
meta_cols = [id_col, lat_col, lon_col]
for c in [block_col, village_col, formation_col]:
    if c and c not in meta_cols:
        meta_cols.append(c)

if qc_col:
    meta_cols.append(qc_col)

wells = df[meta_cols].copy()
wells = wells.dropna(subset=[lat_col, lon_col])

# Use the first non-null metadata value per well.
agg = {}
for c in meta_cols:
    if c != id_col:
        agg[c] = "first"

wells = wells.groupby(id_col, dropna=False).agg(agg).reset_index()

# Check coordinate spread for repeated well IDs.
coord_spread = (
    df.dropna(subset=[lat_col, lon_col])
      .groupby(id_col)
      .agg(
          Latitude_min=(lat_col, "min"),
          Latitude_max=(lat_col, "max"),
          Longitude_min=(lon_col, "min"),
          Longitude_max=(lon_col, "max")
      )
      .reset_index()
)

coord_spread["Latitude_spread"] = (
    coord_spread["Latitude_max"] - coord_spread["Latitude_min"]
)
coord_spread["Longitude_spread"] = (
    coord_spread["Longitude_max"] - coord_spread["Longitude_min"]
)

wells = wells.merge(coord_spread, on=id_col, how="left")

wells["Coordinate_Consistency"] = np.where(
    (wells["Latitude_spread"] > 0.001) |
    (wells["Longitude_spread"] > 0.001),
    "REVIEW_MULTIPLE_COORDINATES",
    "CONSISTENT"
)

# ------------------------------------------------------------
# 5. Well distribution by block and formation
# ------------------------------------------------------------
if block_col:
    block_summary = (
        wells.groupby(block_col, dropna=False)
        .agg(
            Unique_Wells=(id_col, "nunique"),
            Mean_Latitude=(lat_col, "mean"),
            Mean_Longitude=(lon_col, "mean")
        )
        .reset_index()
        .sort_values("Unique_Wells", ascending=False)
    )
else:
    block_summary = pd.DataFrame(
        {"Note": ["Block column not found in source table"]}
    )

if formation_col:
    formation_summary = (
        wells.groupby(formation_col, dropna=False)
        .agg(Unique_Wells=(id_col, "nunique"))
        .reset_index()
        .sort_values("Unique_Wells", ascending=False)
    )
else:
    formation_summary = pd.DataFrame(
        {"Note": ["Hydrogeological formation column not found"]}
    )

# ------------------------------------------------------------
# 6. Save outputs
# ------------------------------------------------------------
wells.to_csv(
    os.path.join(OUT_DIR, "Step2_Well_Spatial_Database.csv"),
    index=False
)

block_summary.to_csv(
    os.path.join(OUT_DIR, "Step2_Block_Well_Distribution.csv"),
    index=False
)

formation_summary.to_csv(
    os.path.join(OUT_DIR, "Step2_Aquifer_Formation_Distribution.csv"),
    index=False
)

coord_spread.to_csv(
    os.path.join(OUT_DIR, "Step2_Well_Coordinate_Consistency.csv"),
    index=False
)

# ------------------------------------------------------------
# 7. Map of observation wells
# ------------------------------------------------------------
plt.figure(figsize=(9, 9))

plt.scatter(
    wells[lon_col],
    wells[lat_col],
    s=18,
    alpha=0.7
)

plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Alwar District: Groundwater Observation Well Locations")
plt.grid(True, alpha=0.3)
plt.axis("equal")

plt.savefig(
    os.path.join(OUT_DIR, "Step2_Alwar_Well_Location_Map.png"),
    dpi=300,
    bbox_inches="tight"
)
plt.show()

# ------------------------------------------------------------
# 8. Block-wise well count chart
# ------------------------------------------------------------
if block_col and "Unique_Wells" in block_summary.columns:
    plot_df = block_summary.dropna(subset=["Unique_Wells"])

    plt.figure(figsize=(11, 7))
    plt.barh(
        plot_df[block_col].astype(str),
        plot_df["Unique_Wells"]
    )
    plt.xlabel("Number of unique wells")
    plt.ylabel("Block")
    plt.title("Groundwater Observation Wells by Block")
    plt.gca().invert_yaxis()
    plt.grid(axis="x", alpha=0.3)

    plt.savefig(
        os.path.join(OUT_DIR, "Step2_Block_Well_Count.png"),
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()

# ------------------------------------------------------------
# 9. Summary and integrity checks
# ------------------------------------------------------------
print("\n" + "="*60)
print("STEP 2A: SPATIAL DATABASE SUMMARY")
print("="*60)

print("Input observation records:", len(df))
print("Unique wells with usable coordinates:", len(wells))
print("Wells with coordinate inconsistency:",
      (wells["Coordinate_Consistency"] != "CONSISTENT").sum())
print("Records outside broad coordinate sanity range:",
      (~valid_coord).sum())

print("\nBLOCK-WISE WELL DISTRIBUTION")
display(block_summary)

print("\nAQUIFER / FORMATION DISTRIBUTION")
display(formation_summary)

print("\nOUTPUT DIRECTORY:")
print(OUT_DIR)

print("\nStep 2A completed.")
